# MIL CNN Experiment Notebook

In [ ]:
# %% ─────────────────────────────────────────────────────────────────────────────
%pip install -q torch torchvision torchaudio tqdm pillow matplotlib


Imports & Configs

In [20]:
# PyTorch & friends
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from torchvision import transforms
import torch.nn.functional as F

# Misc
from pathlib import Path
from PIL import Image, ImageOps
import random, math, json
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from matplotlib import cm
from report_figures.plot_training_curves import plot_training_history
import pandas as pd
import numpy as np
import cv2

from collections import defaultdict
from itertools import chain

import os

In [21]:
# %% ────────────────────────────────────────────────────────────────────────────
# Feel free to tune these
CFG = {
    "train_image_dir": "ICDAR2017_CLaMM_Training_augmented",
    "train_labels_csv": "ICDAR2017_CLaMM_Training/@ICDAR2017_CLaMM_Training.csv",
    "eval_image_dir": "ICDAR2017_CLaMM_evaluation_set",
    "eval_labels_csv": "ICDAR2017_CLaMM_evaluation_set/@ICDAR2017_CLaMM_evaluation_set.csv",
    "eval_split_name": "evaluation_set",
    "eval_split_display_name": "Evaluation set",
    "num_classes": 12,
    "patch_size": 224,             # PxP crops fed to ResNet
    "stride": 224,                 # Stride for patch extraction
    "batch_size": 1,
    "num_workers": max(1, min(4, (os.cpu_count() or 1) // 2)),
    "pin_memory": torch.cuda.is_available(),
    "prefetch_factor": 2,
    "rand_seed": 42,
    "total_epochs": 7,
    "lr_backbone_stage1": 1e-5,
    "lr_head_stage1": 5e-4,
    "weight_decay": 0.04,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "results_dir": "patch_mil_net_results",
    "backbone_name": "resnet34",
    "group_train_variants_into_bag": True,
    "group_eval_variants_into_bag": False,
    # Recommended grouped-bag patch-budget sweep:
    # 128/96, 192/128, 224/160
    "train_top_k_text_patches": 128,
    "train_sample_from_top_text_patches": 192,
    "eval_top_k_text_patches": 24,
    "eval_sample_from_top_text_patches": None,
    "train_variant_suffixes": (
        "",  # original image
        "_inverse",
        "_masked_inv_gray",
        "_inv_masked_gray",
        "_inv_masked_bin",
    ),
}

SCRIPT_NAME_MAP = {}
AUGMENTATION_SUFFIXES = (
    "_inverse",
    "_masked_inv_gray",
    "_inv_masked_gray",
    "_inv_masked_bin",
)
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

random.seed(CFG["rand_seed"])
np.random.seed(CFG["rand_seed"])
torch.manual_seed(CFG["rand_seed"])

os.makedirs(CFG["results_dir"], exist_ok=True)

label2id = {}
id2label = {}



Dataset

In [22]:
def normalize_page_stem(stem: str) -> str:
    for suffix in AUGMENTATION_SUFFIXES:
        if stem.endswith(suffix):
            return stem[: -len(suffix)]
    return stem


def extract_variant_suffix(stem: str) -> str:
    for suffix in AUGMENTATION_SUFFIXES:
        if stem.endswith(suffix):
            return suffix
    return ""


def resolve_train_variant_suffixes(cfg):
    selected = cfg.get("train_variant_suffixes", ("", *AUGMENTATION_SUFFIXES))
    allowed = set(selected)
    valid = {"", *AUGMENTATION_SUFFIXES}
    unknown = sorted(allowed - valid)
    if unknown:
        raise ValueError(
            f"Unknown entries in CFG['train_variant_suffixes']: {unknown}. "
            f"Valid options are: {sorted(valid)}"
        )
    return allowed


def describe_train_variant_selection(selected_suffixes):
    ordered = [suffix or "<original>" for suffix in ["", *AUGMENTATION_SUFFIXES] if suffix in selected_suffixes]
    return ordered


def read_label_csv(csv_path: Path, filename_col: str, label_col: str):
    df = pd.read_csv(csv_path, sep=";")
    lookup = {}
    for _, row in df.iterrows():
        filename = str(row[filename_col]).strip()
        stem = Path(filename).stem
        script_code = int(row[label_col])
        lookup[stem] = SCRIPT_NAME_MAP.get(script_code, f"script_{script_code}")
    return lookup


def collect_image_records(image_dir: Path, label_lookup, allowed_variant_suffixes=None):
    records = []
    for image_path in sorted(image_dir.iterdir()):
        if image_path.suffix.lower() not in IMAGE_EXTS:
            continue

        variant_suffix = extract_variant_suffix(image_path.stem)
        if allowed_variant_suffixes is not None and variant_suffix not in allowed_variant_suffixes:
            continue

        base_stem = normalize_page_stem(image_path.stem)
        label_name = label_lookup.get(base_stem)
        if label_name is None:
            continue

        records.append(
            {
                "image_path": str(image_path),
                "page_id": base_stem,
                "variant_id": image_path.stem,
                "variant_suffix": variant_suffix,
                "label_name": label_name,
                "is_augmented": image_path.stem != base_stem,
            }
        )
    return records


def build_bags_from_records(records, label2id, group_variants: bool):
    if not records:
        return []

    if not group_variants:
        bags = []
        for rec in records:
            bags.append(
                {
                    "image_paths": [rec["image_path"]],
                    "label": label2id[rec["label_name"]],
                    "page_key": rec["page_id"],
                    "script_label": rec["label_name"],
                    "variant_ids": [rec["variant_id"]],
                    "is_grouped_bag": False,
                }
            )
        return bags

    grouped = defaultdict(list)
    for rec in records:
        grouped[rec["page_id"]].append(rec)

    bags = []
    for page_id in sorted(grouped):
        variants = sorted(grouped[page_id], key=lambda r: (r["is_augmented"], r["variant_id"]))
        bags.append(
            {
                "image_paths": [rec["image_path"] for rec in variants],
                "label": label2id[variants[0]["label_name"]],
                "page_key": page_id,
                "script_label": variants[0]["label_name"],
                "variant_ids": [rec["variant_id"] for rec in variants],
                "is_grouped_bag": True,
            }
        )
    return bags

In [23]:


def make_image_transform(train: bool = True):
    """
    Light augmentations on the FULL region image.
    Does NOT resize the full image. Patchification handles the size.
    """
    tfs = []

    if train:
        # Small rotation (very mild to avoid distorting Hebrew letters)
        tfs.append(
            transforms.RandomApply([
                transforms.RandomRotation(
                    degrees=2,
                    fill=0
                )
            ], p=0.5)
        )

        # Small affine jitter
        tfs.append(
            transforms.RandomApply([
                transforms.RandomAffine(
                    degrees=0,
                    translate=(0.02, 0.02),
                    scale=(0.95, 1.05),
                    shear=0,
                    fill=0
                )
            ], p=0.5)
        )

        # Light photometric jitter
        tfs.append(
            transforms.RandomApply([
                transforms.ColorJitter(
                    brightness=0.1,
                    contrast=0.1,
                )
            ], p=0.5)
        )

        # Light blur
        tfs.append(
            transforms.RandomApply([
                transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))
            ], p=0.3)
        )

    # IMPORTANT: do NOT resize image here
    return transforms.Compose(tfs)


def make_patch_transform():
    """
    Transform applied per 224x224 patch.
    Converts to tensor and normalizes for ResNet / ImageNet.
    """
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

In [24]:


class PatchifyBagDataset(Dataset):
    """
    Bag-level dataset with *online* patchification.

    Each item:
        x_bag: [P, 3, patch_size, patch_size]
        y    : scalar label (long)
    """
    def __init__(
        self,
        bags,
        image_transform,
        patch_transform,
        patch_size: int = 224,
        stride: int | None = None,
        top_k_text_patches: int | None = None,
        sample_from_top_text_patches: int | None = None,
        seed: int = 42,
    ):
        """
        Parameters
        ----------
        bags : list[dict]
            Output of build_bags(). Each dict:
              - "image_paths": list[str]
              - "label": int
        image_transform : callable
            Applied to full region image before patchifying.
        patch_transform : callable
            Applied to each cropped patch.
        patch_size : int
            Patch size in pixels (patch_size x patch_size).
        stride : int or None
            Stride for the patch grid. If None, uses stride = patch_size
            (non-overlapping grid). You can set smaller stride for overlap.
        top_k_text_patches : int or None
            Final number of patches returned per bag when patch count must be capped.
        sample_from_top_text_patches : int or None
            If set and larger than top_k_text_patches, first rank patches by an
            Otsu-based, inversion-robust text-to-background ratio, keep the top-M
            pool, then sample k patches from that pool without replacement.
            This adds diversity while still biasing toward text-rich patches.
        seed : int
            Random seed used for patch sampling / oversampling.
        """
        self.bags = bags
        self.image_transform = image_transform
        self.patch_transform = patch_transform
        self.patch_size = patch_size
        self.stride = stride if stride is not None else patch_size
        self.top_k_text_patches = top_k_text_patches
        self.sample_from_top_text_patches = sample_from_top_text_patches
        self.rng = random.Random(seed)

        if self.sample_from_top_text_patches is not None:
            if self.top_k_text_patches is None:
                raise ValueError("sample_from_top_text_patches requires top_k_text_patches to be set")
            if self.sample_from_top_text_patches < self.top_k_text_patches:
                raise ValueError("sample_from_top_text_patches must be >= top_k_text_patches")

    def _enumerate_patch_specs(self, path: str):
        with Image.open(path) as img:
            w, h = img.size

        ps = self.patch_size
        st = self.stride

        if w < ps or h < ps:
            return [(path, None)]

        specs = []
        for top in range(0, h - ps + 1, st):
            for left in range(0, w - ps + 1, st):
                specs.append((path, (left, top, left + ps, top + ps)))
        return specs

    def __len__(self):
        return len(self.bags)

    def _score_patch_text_amount(self, patch: Image.Image) -> float:
        gray = np.array(patch.convert("L"), dtype=np.uint8)
        _, bin_img = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        white_ratio = float((bin_img == 255).mean())
        black_ratio = 1.0 - white_ratio
        return float(min(white_ratio, black_ratio))

    def _crop_patch(self, img: Image.Image, crop_box):
        if crop_box is None:
            return img.resize((self.patch_size, self.patch_size))
        return img.crop(crop_box)

    def _rank_patch_specs_by_text(self, patch_specs):
        image_cache = {}
        scored = []

        for idx, (path, crop_box) in enumerate(patch_specs):
            if path not in image_cache:
                with Image.open(path) as img:
                    image_cache[path] = img.convert("RGB").copy()

            patch = self._crop_patch(image_cache[path], crop_box)
            score = self._score_patch_text_amount(patch)
            scored.append((score, idx, (path, crop_box)))

        scored.sort(key=lambda item: (-item[0], item[1]))
        return [spec for _, _, spec in scored]

    def _select_patch_specs(self, patch_specs):
        if self.top_k_text_patches is None:
            return patch_specs

        n = len(patch_specs)
        if n == 0:
            raise RuntimeError("No patches produced for a bag; check image sizes / transforms.")

        if n > self.top_k_text_patches:
            ranked_patch_specs = self._rank_patch_specs_by_text(patch_specs)
            if self.sample_from_top_text_patches is None:
                return ranked_patch_specs[:self.top_k_text_patches]

            pool_size = min(len(ranked_patch_specs), self.sample_from_top_text_patches)
            candidate_pool = ranked_patch_specs[:pool_size]
            selected_idxs = sorted(self.rng.sample(range(pool_size), self.top_k_text_patches))
            return [candidate_pool[i] for i in selected_idxs]

        if n < self.top_k_text_patches:
            extra = [self.rng.choice(patch_specs) for _ in range(self.top_k_text_patches - n)]
            return patch_specs + extra
        return patch_specs

    def __getitem__(self, idx):
        bag_info = self.bags[idx]
        image_paths = bag_info["image_paths"]
        label = int(bag_info["label"])

        patch_specs = []
        for path in image_paths:
            patch_specs.extend(self._enumerate_patch_specs(path))

        if len(patch_specs) == 0:
            raise RuntimeError(f"No patches produced for bag index {idx} (image_paths={image_paths}).")

        patch_specs = self._select_patch_specs(patch_specs)

        selected_by_path = defaultdict(list)
        for path, crop_box in patch_specs:
            selected_by_path[path].append(crop_box)

        all_patches = []
        for path in image_paths:
            crop_boxes = selected_by_path.get(path)
            if not crop_boxes:
                continue

            with Image.open(path) as img:
                img = img.convert("RGB")
                img = self.image_transform(img)

                for crop_box in crop_boxes:
                    patch = self._crop_patch(img, crop_box)
                    all_patches.append(self.patch_transform(patch))

        x_bag = torch.stack(all_patches, dim=0)  # [P, 3, patch_size, patch_size]
        y = torch.tensor(label, dtype=torch.long)

        return x_bag, y


Model

In [25]:

def make_resnet_backbone(name: str = "resnet18", pretrained: bool = True, cut_last: int = 2):
    """
    Build a torchvision ResNet backbone and cut off the last layers.

    Parameters
    ----------
    name : {"resnet18", "resnet34", "resnet50"}
    pretrained : bool
        If True, use ImageNet weights.
    cut_last : int
        Number of trailing modules to drop. With cut_last=2, we drop
        avgpool and fc, keeping conv1..layer4.

    Returns
    -------
    backbone : nn.Sequential
        Feature extractor (conv1..layer4).
    out_dim : int
        Feature dimension after global average pooling.
    """
    if name == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        base = models.resnet18(weights=weights)
    elif name == "resnet34":
        weights = models.ResNet34_Weights.DEFAULT if pretrained else None
        base = models.resnet34(weights=weights)
    elif name == "resnet50":
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        base = models.resnet50(weights=weights)
    else:
        raise ValueError(f"Unsupported backbone: {name}")

    # children: conv1, bn1, relu, maxpool, layer1–4, avgpool, fc
    backbone = nn.Sequential(*list(base.children())[:-cut_last])
    out_dim = base.fc.in_features  # 512 for resnet18/34, 2048 for resnet50
    return backbone, out_dim


In [26]:
class PatchMILNet(nn.Module):
    """
    MIL network with configurable ResNet backbone.

    backbone_name ∈ {"resnet18", "resnet34", "resnet50"}.
    For your “smaller encoder” baseline, use "resnet18".
    """
    def __init__(
        self,
        num_classes: int,
        backbone_name: str = "resnet18",
        hidden: int = 256,
        dropout: float = 0.3,
        pretrained: bool = True,
    ):
        super().__init__()

        # Backbone: conv1..layer4 (no avgpool, no fc)
        self.backbone, self.backbone_dim = make_resnet_backbone(
            name=backbone_name,
            pretrained=pretrained,
            cut_last=2,
        )

        # Global pooling to get one feature vector per patch
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Dropout on pooled features
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        # Patch classifier head
        self.cls = nn.Linear(self.backbone_dim, num_classes)

        # Attention head
        self.att = nn.Sequential(
            nn.Linear(self.backbone_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, 1),
        )

    def forward(self, x: torch.Tensor):
        """
        x: [B, P, 3, H, W]
        """
        B, P, C, H, W = x.shape
        x = x.view(B * P, C, H, W)          # [B*P, 3, H, W]

        feats = self.backbone(x)            # [B*P, D, h, w]
        feats = self.global_pool(feats).squeeze(-1).squeeze(-1)  # [B*P, D]
        feats = self.dropout(feats)

        logits_patch = self.cls(feats)      # [B*P, C]
        att_scores   = self.att(feats)      # [B*P, 1]

        logits_patch = logits_patch.view(B, P, -1)  # [B, P, C]
        att_scores   = att_scores.view(B, P)        # [B, P]

        att_w = torch.softmax(att_scores, dim=1).unsqueeze(-1)   # [B, P, 1]
        logits_page = (att_w * logits_patch).sum(dim=1)          # [B, C]

        return logits_page, logits_patch, att_w


In [27]:
def build_patch_mil_model(cfg, device: str, for_training: bool = True):
    """
    Build PatchMILNet.

    Training mode defaults to ImageNet init.
    Eval / visualization mode recreates the architecture without ImageNet init,
    because the full MIL checkpoint will be loaded immediately afterward.
    """
    model = PatchMILNet(
        num_classes=cfg["num_classes"],
        backbone_name=cfg["backbone_name"],
        hidden=256,
        dropout=0.3,
        pretrained=for_training,
    ).to(device)
    return model


Train!

In [28]:
train_label_lookup = read_label_csv(
    Path(CFG["train_labels_csv"]),
    filename_col="FILENAME",
    label_col="SCRIPT_TYPE",
)
eval_label_lookup = read_label_csv(
    Path(CFG["eval_labels_csv"]),
    filename_col="FILE_NAME",
    label_col="Script_type_ICDAR2017",
)

all_label_names = sorted(set(train_label_lookup.values()) | set(eval_label_lookup.values()))
label2id = {label: idx for idx, label in enumerate(all_label_names)}
id2label = {idx: label for label, idx in label2id.items()}
CFG["num_classes"] = len(label2id)

allowed_train_variant_suffixes = resolve_train_variant_suffixes(CFG)
selected_train_variants = describe_train_variant_selection(allowed_train_variant_suffixes)

train_records = collect_image_records(
    Path(CFG["train_image_dir"]),
    train_label_lookup,
    allowed_variant_suffixes=allowed_train_variant_suffixes,
)
eval_records = collect_image_records(Path(CFG["eval_image_dir"]), eval_label_lookup)

train_group_mode = bool(CFG.get("group_train_variants_into_bag", False))
eval_group_mode = bool(CFG.get("group_eval_variants_into_bag", False))

# Keep training bags aligned with evaluation by default: one real page per bag.
# Augmented page variants still contribute as separate training samples.
train_bags = build_bags_from_records(train_records, label2id, group_variants=train_group_mode)
val_bags = build_bags_from_records(eval_records, label2id, group_variants=eval_group_mode)

patch_size = CFG["patch_size"]
stride = CFG["stride"]

train_img_tf = make_image_transform(train=True)
val_img_tf = make_image_transform(train=False)
patch_tf = make_patch_transform()

train_ds = PatchifyBagDataset(
    train_bags,
    image_transform=train_img_tf,
    patch_transform=patch_tf,
    patch_size=patch_size,
    stride=stride,
    top_k_text_patches=CFG["train_top_k_text_patches"],
    sample_from_top_text_patches=CFG["train_sample_from_top_text_patches"],
)

val_ds = PatchifyBagDataset(
    val_bags,
    image_transform=val_img_tf,
    patch_transform=patch_tf,
    patch_size=patch_size,
    stride=stride,
    top_k_text_patches=CFG["eval_top_k_text_patches"],
    sample_from_top_text_patches=CFG["eval_sample_from_top_text_patches"],
)

loader_kwargs = {
    "batch_size": CFG["batch_size"],
    "num_workers": CFG["num_workers"],
    "pin_memory": CFG["pin_memory"],
}
if CFG["num_workers"] > 0:
    loader_kwargs["persistent_workers"] = True
    loader_kwargs["prefetch_factor"] = CFG["prefetch_factor"]

train_dl = DataLoader(train_ds, shuffle=True, **loader_kwargs)
val_dl = DataLoader(val_ds, shuffle=False, **loader_kwargs)

print("Label mapping:", label2id)
print(f"Selected train variants:     {selected_train_variants}")
print(f"Train records (selected):    {len(train_records)}")
print(f"Eval records  ({CFG['eval_split_display_name']}): {len(eval_records)}")
train_bag_mode_label = "grouped page variants" if train_group_mode else "single pages / variants as separate bags"
eval_bag_mode_label = "grouped page variants" if eval_group_mode else "single pages / variants as separate bags"
print(f"Train bags ({train_bag_mode_label}):  {len(train_bags)}")
print(f"Eval bags  ({eval_bag_mode_label}):   {len(val_bags)}")



Label mapping: {'script_1': 0, 'script_10': 1, 'script_11': 2, 'script_12': 3, 'script_2': 4, 'script_3': 5, 'script_4': 6, 'script_5': 7, 'script_6': 8, 'script_7': 9, 'script_8': 10, 'script_9': 11}
Selected train variants:     ['<original>', '_inverse', '_masked_inv_gray', '_inv_masked_gray', '_inv_masked_bin']
Train records (selected):    17700
Eval records  (Evaluation set): 1999
Train bags (grouped page variants):  3540
Eval bags  (single pages / variants as separate bags):   1999


In [29]:
def set_backbone_requires_grad(model: PatchMILNet, requires_grad: bool):
    for p in model.backbone.parameters():
        p.requires_grad = requires_grad


def make_optimizer(model, lr_backbone: float, lr_head: float, weight_decay: float):
    """
    Single-stage training: backbone unfrozen, but with smaller LR than head.
    """
    params = [
        {"params": model.backbone.parameters(), "lr": lr_backbone},
        {"params": chain(model.cls.parameters(), model.att.parameters()), "lr": lr_head},
    ]
    return torch.optim.AdamW(params, weight_decay=weight_decay)



In [30]:

def train_epoch(model, loader, crit, opt, device, use_amp: bool = False):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    for bags, labels in pbar:        # bags: [B, P, 3, H, W]
        bags   = bags.to(device, non_blocking=CFG["pin_memory"])
        labels = labels.to(device, non_blocking=CFG["pin_memory"])

        opt.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", enabled=use_amp):
            logits_page, _, _ = model(bags)        # [B, C]
            loss = crit(logits_page, labels)

        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        preds = logits_page.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += batch_size

        pbar.set_description(
            f"train loss {total_loss/total:.4f} | acc {correct/total:.3f}"
        )

    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, crit, device, use_amp: bool = False):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for bags, labels in loader:
        bags   = bags.to(device, non_blocking=CFG["pin_memory"])
        labels = labels.to(device, non_blocking=CFG["pin_memory"])

        with torch.amp.autocast(device_type="cuda", enabled=use_amp):
            logits_page, _, _ = model(bags)
            loss = crit(logits_page, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        preds = logits_page.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += batch_size

    return total_loss / total, correct / total


In [12]:
device = CFG["device"]

TOTAL_EPOCHS = CFG["total_epochs"]
LR_BACKBONE_STAGE1 = CFG["lr_backbone_stage1"]
LR_HEAD_STAGE1 = CFG["lr_head_stage1"]
WEIGHT_DECAY = CFG["weight_decay"]

model = build_patch_mil_model(CFG, device=device, for_training=True)

criterion = nn.CrossEntropyLoss()
use_amp = device == "cuda"
if device == "cuda":
    torch.set_float32_matmul_precision("high")

best_val_acc = 0.0
best_val_loss = float("inf")
best_ckpt_path = CFG["results_dir"] + "/best_mil_patchnet.pth"
history = {
    "epoch": [],
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}
history_path = Path(CFG["results_dir"]) / "training_history.json"

print(f"Training for {TOTAL_EPOCHS} epochs with a single unfrozen stage")

set_backbone_requires_grad(model, True)
optimizer = make_optimizer(
    model,
    lr_backbone=LR_BACKBONE_STAGE1,
    lr_head=LR_HEAD_STAGE1,
    weight_decay=WEIGHT_DECAY,
)

for epoch in range(1, TOTAL_EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_dl, criterion, optimizer, device, use_amp=use_amp)
    val_loss, val_acc = eval_epoch(model, val_dl, criterion, device, use_amp=use_amp)

    history["epoch"].append(epoch)
    history["train_loss"].append(float(tr_loss))
    history["train_acc"].append(float(tr_acc))
    history["val_loss"].append(float(val_loss))
    history["val_acc"].append(float(val_acc))

    print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} | train acc {tr_acc:.3f}")
    print(f"Epoch {epoch:02d} |   val loss {val_loss:.4f} |   val acc {val_acc:.3f}")

    is_better = (val_acc > best_val_acc) or (
        (val_acc == best_val_acc) and (val_loss < best_val_loss)
    )

    if is_better:
        best_val_acc = val_acc
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_ckpt_path)
        print(f"  -> New best model saved: acc={best_val_acc:.3f}, loss={best_val_loss:.4f}")

with open(history_path, "w") as f:
    json.dump(history, f, indent=2)

best_model_info = {
    "best_model_checkpoint": best_ckpt_path,
    "best_metric_accuracy": best_val_acc,
    "best_metric_loss": best_val_loss,
    "training_history_json": str(history_path),
    "eval_split": CFG["eval_split_display_name"],
    "eval_split_key": CFG["eval_split_name"],
    "num_labels": CFG["num_classes"],
    "backbone_name": CFG["backbone_name"],
    "train_group_variants_into_bag": bool(CFG.get("group_train_variants_into_bag", False)),
    "train_top_k_text_patches": CFG["train_top_k_text_patches"],
    "train_sample_from_top_text_patches": CFG["train_sample_from_top_text_patches"],
    "eval_top_k_text_patches": CFG["eval_top_k_text_patches"],
    "eval_sample_from_top_text_patches": CFG["eval_sample_from_top_text_patches"],
    "train_variant_suffixes": list(CFG.get("train_variant_suffixes", ())),
}

with open(Path(CFG["results_dir"]) / "best_model_info.json", "w") as f:
    json.dump(best_model_info, f, indent=2)

print("Saved training history to:", history_path)
print("Saved best model info to:", Path(CFG["results_dir"]) / "best_model_info.json")



Training for 7 epochs with a single unfrozen stage


train loss 1.4580 | acc 0.456: 100%|██████████| 3540/3540 [23:18<00:00,  2.53it/s]


Epoch 01 | train loss 1.4580 | train acc 0.456
Epoch 01 |   val loss 1.3071 |   val acc 0.515
  -> New best model saved: acc=0.515, loss=1.3071


train loss 0.8720 | acc 0.657: 100%|██████████| 3540/3540 [23:18<00:00,  2.53it/s]


Epoch 02 | train loss 0.8720 | train acc 0.657
Epoch 02 |   val loss 1.2609 |   val acc 0.551
  -> New best model saved: acc=0.551, loss=1.2609


train loss 0.6783 | acc 0.741: 100%|██████████| 3540/3540 [23:24<00:00,  2.52it/s]


... (5 outputs left out)

train loss 0.4208 | acc 0.842: 100%|██████████| 3540/3540 [22:54<00:00,  2.58it/s]


Epoch 06 | train loss 0.4208 | train acc 0.842
Epoch 06 |   val loss 1.1899 |   val acc 0.657


train loss 0.3802 | acc 0.854: 100%|██████████| 3540/3540 [23:26<00:00,  2.52it/s]


Epoch 07 | train loss 0.3802 | train acc 0.854
Epoch 07 |   val loss 0.8428 |   val acc 0.742
Saved training history to: patch_mil_net_results/training_history.json
Saved best model info to: patch_mil_net_results/best_model_info.json


In [13]:
png_path, pdf_path, csv_path = plot_training_history(
    history=history,
    output_dir="report_figures",
    stem="mil_cnn_train_val_curves",
    title="MIL CNN Training and Validation Curves",
)

print("Saved:", png_path)
print("Saved:", pdf_path)
print("Saved:", csv_path)


Saved: report_figures/mil_cnn_train_val_curves.png
Saved: report_figures/mil_cnn_train_val_curves.pdf
Saved: report_figures/mil_cnn_train_val_curves.csv


## Evaluate the Evaluation set with the best checkpoint

In [31]:
patch_size = CFG["patch_size"]
stride = CFG["stride"]

# Reuse the same Evaluation set bags/loader used during training.
# top_k_text_patches=400 already covers the full page for this split (observed max is 88).
eval_bags = val_bags
eval_ds = val_ds
eval_dl = val_dl



In [32]:

device = CFG["device"]

checkpoint_path = CFG["results_dir"] + "/best_mil_patchnet.pth"

# Recreate the model architecture exactly as in training
model = build_patch_mil_model(CFG, device=device, for_training=False)
state = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(state)
model.eval()


PatchMILNet(
  (backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=

In [33]:
def build_confusion_tables(df_all: pd.DataFrame, id2label):
    label_ids = sorted(id2label)
    label_names = [id2label[idx] for idx in label_ids]
    conf = pd.crosstab(
        pd.Categorical(df_all["true_label"], categories=label_names),
        pd.Categorical(df_all["pred_label"], categories=label_names),
        dropna=False,
    )
    conf.index.name = "true_label"
    conf.columns.name = "pred_label"

    conf_norm = conf.div(conf.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)
    return conf, conf_norm


def build_per_class_metrics(df_all: pd.DataFrame, id2label):
    conf, _ = build_confusion_tables(df_all, id2label)
    rows = []

    for label_name in conf.index:
        tp = int(conf.loc[label_name, label_name])
        support = int(conf.loc[label_name].sum())
        pred_total = int(conf[label_name].sum())
        fn = support - tp
        fp = pred_total - tp

        precision = tp / pred_total if pred_total else 0.0
        recall = tp / support if support else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

        rows.append(
            {
                "class": label_name,
                "support": support,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "false_positives": fp,
                "false_negatives": fn,
            }
        )

    return pd.DataFrame(rows)


def build_summary_table(task_metrics: dict):
    cols = [
        "eval_split",
        "backbone_name",
        "n_eval_pages",
        "loss",
        "accuracy",
        "balanced_accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
    ]
    return pd.DataFrame([{col: task_metrics[col] for col in cols}])


def plot_confusion_matrix(conf_norm: pd.DataFrame, output_path: Path, title: str):
    fig, ax = plt.subplots(figsize=(10.5, 8.5))
    im = ax.imshow(conf_norm.values, cmap="Blues", vmin=0.0, vmax=1.0)

    ax.set_xticks(np.arange(len(conf_norm.columns)))
    ax.set_yticks(np.arange(len(conf_norm.index)))
    ax.set_xticklabels(conf_norm.columns, rotation=45, ha="right")
    ax.set_yticklabels(conf_norm.index)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(title)

    threshold = 0.5
    for i in range(conf_norm.shape[0]):
        for j in range(conf_norm.shape[1]):
            value = conf_norm.iat[i, j]
            ax.text(
                j,
                i,
                f"{value:.2f}",
                ha="center",
                va="center",
                color="white" if value > threshold else "#1f1f1f",
                fontsize=8,
            )

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Normalized count")
    fig.tight_layout()

    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(output_path.with_suffix(".pdf"), bbox_inches="tight", facecolor="white")
    plt.close(fig)


def save_split_outputs(
    split_name: str,
    debug_df: pd.DataFrame,
    summary_df: pd.DataFrame,
    per_class_df: pd.DataFrame,
    confusion_df: pd.DataFrame,
    confusion_norm_df: pd.DataFrame,
    base_dir: Path,
    metrics_dict=None,
    split_display_name=None,
):
    display_name = split_display_name or split_name
    metrics_dir = base_dir / "metrics"
    debug_dir = base_dir / "predictions" / "debug"
    report_dir = base_dir / "report"

    metrics_dir.mkdir(parents=True, exist_ok=True)
    debug_dir.mkdir(parents=True, exist_ok=True)
    report_dir.mkdir(parents=True, exist_ok=True)

    debug_csv = debug_dir / f"{split_name}_predictions.csv"
    summary_csv = report_dir / f"{split_name}_summary_table.csv"
    summary_tex = report_dir / f"{split_name}_summary_table.tex"
    per_class_csv = report_dir / f"{split_name}_per_class_table.csv"
    per_class_tex = report_dir / f"{split_name}_per_class_table.tex"
    confusion_csv = report_dir / f"{split_name}_confusion_matrix.csv"
    confusion_norm_csv = report_dir / f"{split_name}_confusion_matrix_normalized.csv"
    confusion_png = report_dir / f"{split_name}_confusion_matrix.png"

    debug_df.to_csv(debug_csv, index=False)
    summary_df.to_csv(summary_csv, index=False)
    summary_df.to_latex(summary_tex, index=False, float_format="%.3f")
    per_class_df.to_csv(per_class_csv, index=False)
    per_class_df.to_latex(per_class_tex, index=False, float_format="%.3f")
    confusion_df.to_csv(confusion_csv)
    confusion_norm_df.to_csv(confusion_norm_csv)

    plot_confusion_matrix(
        confusion_norm_df,
        output_path=confusion_png,
        title=f"{display_name} Normalized Confusion Matrix",
    )

    saved_paths = {
        "debug_csv": debug_csv,
        "summary_csv": summary_csv,
        "summary_tex": summary_tex,
        "per_class_csv": per_class_csv,
        "per_class_tex": per_class_tex,
        "confusion_csv": confusion_csv,
        "confusion_norm_csv": confusion_norm_csv,
        "confusion_png": confusion_png,
        "confusion_pdf": confusion_png.with_suffix(".pdf"),
    }

    if metrics_dict is not None:
        metrics_path = metrics_dir / f"{split_name}_summary.json"
        with open(metrics_path, "w") as f:
            json.dump(metrics_dict, f, indent=2)
        saved_paths["metrics_json"] = metrics_path

    print(f"{display_name} artifacts:")
    print(f"  Metrics JSON: {saved_paths.get('metrics_json', 'not saved')}")
    print(f"  Debug CSV: {saved_paths['debug_csv']}")
    print(f"  Summary table: {saved_paths['summary_tex']}")
    print(f"  Per-class table: {saved_paths['per_class_tex']}")
    print(f"  Confusion matrix figure: {saved_paths['confusion_png']}")

    return saved_paths


@torch.no_grad()
def infer_on_ds(model, loader, bags, device, id2label):
    """
    Run inference on the same evaluation loader used during training.
    """
    model.eval()
    records = []
    label_names = [id2label[idx] for idx in sorted(id2label)]

    bag_ptr = 0

    for xb, _ in loader:
        xb = xb.to(device)
        B, P = xb.shape[0], xb.shape[1]

        logits_page, _, _ = model(xb)
        probs = F.softmax(logits_page, dim=1)
        preds = probs.argmax(dim=1)
        confidences = probs.max(dim=1).values

        for b in range(B):
            bag_info = bags[bag_ptr]
            bag_ptr += 1

            page_key = bag_info.get("page_key", f"bag_{bag_ptr-1}")
            true_id = int(bag_info["label"])
            pred_id = int(preds[b].item())

            row = {
                "page_key": page_key,
                "true_label_int": true_id,
                "pred_label_int": pred_id,
                "pred_label": id2label[pred_id],
                "true_label": id2label[true_id],
                "confidence": float(confidences[b].item()),
                "num_patches": int(P),
            }
            for label_idx, label_name in enumerate(label_names):
                row[f"p({label_name})"] = float(probs[b, label_idx].item())

            records.append(row)

    df_all = pd.DataFrame(records)
    df_all = df_all.sort_values("page_key").reset_index(drop=True)
    df_all["correct"] = df_all["pred_label_int"] == df_all["true_label_int"]

    accuracy = float(df_all["correct"].mean()) if len(df_all) else float("nan")
    return accuracy, df_all


criterion = nn.CrossEntropyLoss()
task_loss, task_acc = eval_epoch(model, eval_dl, criterion, device)
task_acc_from_preds, task_df_all = infer_on_ds(
    model,
    eval_dl,
    eval_bags,
    device=CFG["device"],
    id2label=id2label,
)

per_class_df = build_per_class_metrics(task_df_all, id2label)
confusion_df, confusion_norm_df = build_confusion_tables(task_df_all, id2label)

macro_precision = float(per_class_df["precision"].mean())
macro_recall = float(per_class_df["recall"].mean())
macro_f1 = float(per_class_df["f1"].mean())
balanced_accuracy = macro_recall

eval_split_display_name = CFG["eval_split_display_name"]
print(f"{eval_split_display_name} accuracy (page-level): {task_acc:.4f}")
print(f"{eval_split_display_name} loss (page-level CE):  {task_loss:.4f}")
print(f"{eval_split_display_name} macro F1:             {macro_f1:.4f}")

task_metrics = {
    "best_model_checkpoint": checkpoint_path,
    "eval_split": eval_split_display_name,
    "eval_split_key": CFG["eval_split_name"],
    "backbone_name": CFG["backbone_name"],
    "train_group_variants_into_bag": bool(CFG.get("group_train_variants_into_bag", False)),
    "patch_size": patch_size,
    "stride": stride,
    "train_top_k_text_patches": CFG["train_top_k_text_patches"],
    "train_sample_from_top_text_patches": CFG["train_sample_from_top_text_patches"],
    "eval_top_k_text_patches": getattr(eval_ds, "top_k_text_patches", None),
    "eval_sample_from_top_text_patches": getattr(eval_ds, "sample_from_top_text_patches", None),
    "n_eval_pages": int(len(eval_bags)),
    "label_mapping": id2label,
    "loss": float(task_loss),
    "accuracy": float(task_acc),
    "accuracy_from_predictions": float(task_acc_from_preds),
    "balanced_accuracy": balanced_accuracy,
    "macro_precision": macro_precision,
    "macro_recall": macro_recall,
    "macro_f1": macro_f1,
}

summary_df = build_summary_table(task_metrics)

task_outputs = save_split_outputs(
    split_name=CFG["eval_split_name"],
    split_display_name=eval_split_display_name,
    debug_df=task_df_all,
    summary_df=summary_df,
    per_class_df=per_class_df,
    confusion_df=confusion_df,
    confusion_norm_df=confusion_norm_df,
    base_dir=Path(CFG["results_dir"]),
    metrics_dict=task_metrics,
)



Evaluation set accuracy (page-level): 0.7554
Evaluation set loss (page-level CE):  0.8229
Evaluation set macro F1:             0.6939
Evaluation set artifacts:
  Metrics JSON: patch_mil_net_results/metrics/evaluation_set_summary.json
  Debug CSV: patch_mil_net_results/predictions/debug/evaluation_set_predictions.csv
  Summary table: patch_mil_net_results/report/evaluation_set_summary_table.tex
  Per-class table: patch_mil_net_results/report/evaluation_set_per_class_table.tex
  Confusion matrix figure: patch_mil_net_results/report/evaluation_set_confusion_matrix.png


Visualize Attention Maps

In [34]:

def compute_attention_for_bag(
    model,
    bag_info,
    patch_size: int = 224,
    stride: int = 224,
    patch_transform=None,
    device: str = "cuda",
):
    """
    Compute MIL attention weights per patch and organize them into
    patch grids per region image.

    Returns
    -------
    per_image_att : list of dicts
        Each dict contains:
            - "image_path": str
            - "att_grid": np.ndarray of shape [n_rows, n_cols]
            - "H": original image height
            - "W": original image width
            - "patch_size": patch size used for MIL attention
            - "stride": stride used for patch extraction
    """
    model.eval()

    if patch_transform is None:
        patch_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ])

    patches = []
    meta = []  # (img_idx, top, left, H, W)

    image_paths = bag_info["image_paths"]

    # 1) Patchify all region images
    for img_idx, path in enumerate(image_paths):
        img = Image.open(path).convert("RGB")
        W, H = img.size

        # simple regular grid; must match training patchify logic
        if W < patch_size or H < patch_size:
            # fallback: single resized patch
            crop = img.resize((patch_size, patch_size))
            patches.append(patch_transform(crop))
            meta.append((img_idx, 0, 0, patch_size, patch_size))
        else:
            for top in range(0, H - patch_size + 1, stride):
                for left in range(0, W - patch_size + 1, stride):
                    crop = img.crop((left, top, left + patch_size, top + patch_size))
                    patches.append(patch_transform(crop))
                    meta.append((img_idx, top, left, H, W))

    if len(patches) == 0:
        raise RuntimeError(f"No patches produced for bag with page_key={bag_info.get('page_key')}")

    # 2) Run through model to get attention weights
    x = torch.stack(patches, dim=0).unsqueeze(0).to(device)  # [1, P, 3, ps, ps]
    with torch.no_grad():
        _, _, att_w = model(x)       # att_w: [1, P, 1]

    att = att_w.squeeze(0).squeeze(-1).cpu().numpy()  # [P]

    # 3) Build per-image attention grids
    img_to_patches = {}
    for i, (img_idx, top, left, H, W) in enumerate(meta):
        if img_idx not in img_to_patches:
            img_to_patches[img_idx] = {
                "coords": [],
                "H": H,
                "W": W,
            }
        img_to_patches[img_idx]["coords"].append((top, left, i))

    per_image_att = []

    for img_idx, info in img_to_patches.items():
        coords = info["coords"]
        H, W = info["H"], info["W"]

        if H < patch_size or W < patch_size:
            n_rows, n_cols = 1, 1
        else:
            n_rows = (H - patch_size) // stride + 1
            n_cols = (W - patch_size) // stride + 1

        grid = np.zeros((n_rows, n_cols), dtype=np.float32)

        for (top, left, patch_i) in coords:
            if H < patch_size or W < patch_size:
                r, c = 0, 0
            else:
                r = top // stride
                c = left // stride
            grid[r, c] = att[patch_i]

        per_image_att.append({
            "image_path": image_paths[img_idx],
            "att_grid": grid,
            "H": H,
            "W": W,
            "patch_size": patch_size,
            "stride": stride,
        })

    return per_image_att


In [35]:

def save_attention_overlay(
    image_path,
    att_grid,
    H,
    W,
    out_path,
    patch_size=224,
    stride=224,
    cmap_name="magma",
    alpha_max=0.78,
    clip_percentiles=(1, 99),
    low_attention_quantile=0.60,
    gamma=1.35,
    base_desaturate=0.72,
    add_colorbar=True,
    draw_patch_grid=False,
    save_original=False,
    save_heatmap_panel=True,
):
    """
    Create a patch-faithful attention overlay that is easier to defend in a paper.

    Key changes relative to a quick debugging overlay:
    - no bilinear smoothing across the page
    - low-attention patches are visually suppressed
    - the manuscript is slightly desaturated so attended regions read clearly
    """

    img = Image.open(image_path).convert("RGB")
    img_np = np.array(img).astype(np.float32) / 255.0

    att = att_grid.astype(np.float32).copy()
    if clip_percentiles is not None:
        lo, hi = np.percentile(att, clip_percentiles)
        if hi > lo:
            att = np.clip(att, lo, hi)

    att_min, att_max = float(att.min()), float(att.max())
    if att_max > att_min:
        att_norm = (att - att_min) / (att_max - att_min)
    else:
        att_norm = np.zeros_like(att, dtype=np.float32)

    if low_attention_quantile is not None:
        keep_floor = float(np.quantile(att_norm, low_attention_quantile))
        att_visible = np.where(att_norm >= keep_floor, att_norm, 0.0)
    else:
        att_visible = att_norm

    overlay_values = np.zeros((H, W), dtype=np.float32)
    overlay_alpha = np.zeros((H, W), dtype=np.float32)

    n_rows, n_cols = att_visible.shape
    for r in range(n_rows):
        for c in range(n_cols):
            top = r * stride
            left = c * stride
            bottom = min(top + patch_size, H)
            right = min(left + patch_size, W)

            val = float(att_visible[r, c])
            overlay_values[top:bottom, left:right] = val
            overlay_alpha[top:bottom, left:right] = (val ** gamma) * alpha_max

    gray = np.repeat(img_np.mean(axis=2, keepdims=True), 3, axis=2)
    base = base_desaturate * gray + (1.0 - base_desaturate) * img_np

    cmap = plt.get_cmap(cmap_name)
    att_color = cmap(overlay_values)[..., :3].astype(np.float32)

    overlay = (1.0 - overlay_alpha[..., None]) * base + overlay_alpha[..., None] * att_color
    overlay = np.clip(overlay, 0.0, 1.0)

    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    if save_original:
        orig_path = os.path.splitext(out_path)[0] + "_original.png"
        img.save(orig_path)

    aspect = H / max(W, 1)
    fig_w = 8
    fig_h = max(4, fig_w * aspect)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.imshow(overlay, interpolation="nearest")

    if draw_patch_grid:
        for x in range(0, W + 1, stride):
            ax.axvline(x - 0.5, color="white", linewidth=0.25, alpha=0.25)
        for y in range(0, H + 1, stride):
            ax.axhline(y - 0.5, color="white", linewidth=0.25, alpha=0.25)

    ax.axis("off")

    if add_colorbar:
        mappable = cm.ScalarMappable(cmap=cmap)
        mappable.set_clim(0.0, 1.0)
        cbar = fig.colorbar(mappable, ax=ax, fraction=0.030, pad=0.018)
        cbar.set_label("Normalized patch attention", rotation=270, labelpad=14)

    fig.tight_layout(pad=0)
    fig.savefig(out_path, dpi=400, bbox_inches="tight", pad_inches=0)
    plt.close(fig)

    if save_heatmap_panel:
        heatmap_path = os.path.splitext(out_path)[0] + "_grid.png"
        fig, ax = plt.subplots(figsize=(6, max(3, 6 * n_rows / max(n_cols, 1))))
        im = ax.imshow(att_norm, cmap=cmap_name, vmin=0.0, vmax=1.0, interpolation="nearest")
        ax.set_xlabel("Patch column")
        ax.set_ylabel("Patch row")
        ax.set_title("Patch-level MIL attention")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        fig.tight_layout()
        fig.savefig(heatmap_path, dpi=300, bbox_inches="tight")
        plt.close(fig)


In [36]:
device = CFG["device"]
patch_size = CFG["patch_size"]
stride = CFG.get("stride", patch_size)
results_dir = CFG["results_dir"]

# Recreate model and load best checkpoint
model = build_patch_mil_model(CFG, device=device, for_training=False)
state = torch.load(results_dir + "/best_mil_patchnet.pth", map_location=device)
model.load_state_dict(state)
model.eval()

n_correct_examples = 2
n_wrong_examples = 1

correct_candidates = (
    task_df_all[task_df_all["correct"]]
    .sort_values(["confidence", "page_key"], ascending=[False, True])
    .head(n_correct_examples)
)

wrong_candidates = (
    task_df_all[~task_df_all["correct"]]
    .sort_values(["confidence", "page_key"], ascending=[False, True])
    .head(n_wrong_examples)
)

selected_examples = pd.concat([correct_candidates, wrong_candidates], ignore_index=True)

if len(correct_candidates) < n_correct_examples:
    print(f"Requested {n_correct_examples} correct pages but found only {len(correct_candidates)}.")
if len(wrong_candidates) < n_wrong_examples:
    print(f"Requested {n_wrong_examples} wrong pages but found only {len(wrong_candidates)}.")

selected_page_keys = selected_examples["page_key"].tolist()
selected_bags = {b.get("page_key"): b for b in eval_bags}
bags_to_visualize = [selected_bags[k] for k in selected_page_keys if k in selected_bags]

print("Selected pages for attention visualization:")
display(selected_examples[["page_key", "true_label", "pred_label", "confidence", "correct", "num_patches"]])

for i, bag_info in enumerate(bags_to_visualize):
    page_key = bag_info.get("page_key", f"bag_{i}")

    per_image_att = compute_attention_for_bag(
        model,
        bag_info,
        patch_size=patch_size,
        stride=stride,
        patch_transform=make_patch_transform(),
        device=device,
    )

    vis_dir = os.path.join(results_dir, "attention_vis_publication", page_key)

    for j, info in enumerate(per_image_att):
        img_path = info["image_path"]
        att_grid = info["att_grid"]
        H, W = info["H"], info["W"]

        stem = os.path.splitext(os.path.basename(img_path))[0]
        out_path = os.path.join(vis_dir, f"{stem}_att_overlay_publication.png")

        save_attention_overlay(
            image_path=img_path,
            att_grid=att_grid,
            H=H,
            W=W,
            out_path=out_path,
            patch_size=info.get("patch_size", patch_size),
            stride=info.get("stride", stride),
            cmap_name="magma",
            alpha_max=0.78,
            clip_percentiles=(1, 99),
            low_attention_quantile=0.60,
            gamma=1.35,
            base_desaturate=0.72,
            add_colorbar=True,
            draw_patch_grid=False,
            save_original=True,
            save_heatmap_panel=True,
        )

    print(f"Saved publication-style attention overlay: {out_path}")



Selected pages for attention visualization:


,page_key,true_label,pred_label,confidence,correct,num_patches
0,B490076101_MS0378_0011,script_10,script_10,1.000000,True,24
1,B490076101_MS0378_0019,script_10,script_10,1.000000,True,24
2,IRHT_P_008107,script_11,script_10,0.999945,False,24


Saved publication-style attention overlay: patch_mil_net_results/attention_vis_publication/B490076101_MS0378_0011/B490076101_MS0378_0011_att_overlay_publication.png
Saved publication-style attention overlay: patch_mil_net_results/attention_vis_publication/B490076101_MS0378_0019/B490076101_MS0378_0019_att_overlay_publication.png
Saved publication-style attention overlay: patch_mil_net_results/attention_vis_publication/IRHT_P_008107/IRHT_P_008107_att_overlay_publication.png
